# **Portfolio:** Mario Casanova — Data Science & Analytics Portfolio
## **Case Study:** A/B Testing, Optimization & Statistical Inference

---
*Note: this is a synthetic experiment, generated by `src/data_generator.py`. The redesigned onboarding flow carries a small true conversion lift (~2 percentage points), placed deliberately near the edge of detectability, because that edge is the interesting part. The observed result turns out significant under a one-sided test but not under a two-sided one, which is exactly the situation where the choice of test, the effect size, and statistical power decide the verdict rather than the data speaking for itself. That is why I pre-register the direction beforehand (so one-sided is defensible on its own terms, not chosen after peeking), report both p-values, and read a marginal result for what it is rather than what I'd like it to be. Assignment is randomized by construction, so the causal reading holds within the simulation — it is not a measurement of a real product.

In [1]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
sns.set_palette("Blues")

### 1. Experiment Data Ingestion (Random Assignment)
Before comparing anything, I check the partition itself: how many users landed in the Control Flow (Classic Onboarding) versus the Variant (Redesigned Onboarding Flow), and whether that split looks like the coin flip it's supposed to be.

In [2]:
df_users = pd.read_csv('../data/hardware_users.csv')
df_subs = pd.read_csv('../data/subscriptions.csv')

df_subs['Has_Subscribed'] = 1
df_ab = pd.merge(df_users, df_subs[['UserID', 'Has_Subscribed']], on='UserID', how='left')
df_ab['Has_Subscribed'] = df_ab['Has_Subscribed'].fillna(0).astype(int)

print("[*] A/B Experiment Sample Summary:\n")
print(df_ab['OnboardingVersion'].value_counts())
df_ab.head()

[*] A/B Experiment Sample Summary:

OnboardingVersion
Control    2515
Variant    2485
Name: count, dtype: int64


,UserID,DeviceType,PurchaseDate,OnboardingVersion,CohortMonth,Has_Subscribed
0,10000,Security Camera,2022-02-20,Control,2022-02,1
1,10001,Smart Doorbell,2022-03-12,Variant,2022-03,0
2,10002,Smart Doorbell,2022-02-17,Variant,2022-02,1
3,10003,Smart Doorbell,2022-01-19,Variant,2022-01,0
4,10004,Security Camera,2022-09-19,Variant,2022-09,0


### 2. Empirical Results (Sample Conversion Rates)
I group the binary outcomes to see the raw, superficial difference first — before the Central Limit Theorem gets a say in whether that difference means anything.

In [3]:
ab_summary = df_ab.groupby('OnboardingVersion').agg(
    Exposures=('UserID', 'count'),
    Conversions=('Has_Subscribed', 'sum'),
    Conversion_Rate=('Has_Subscribed', 'mean')
)

ab_summary['Conversion_Rate_Pct'] = ab_summary['Conversion_Rate'] * 100
ab_summary

,Exposures,Conversions,Conversion_Rate,Conversion_Rate_Pct
OnboardingVersion,,,,
Control,2515,679,0.269980,26.998012
Variant,2485,724,0.291348,29.134809


### 3. Hypothesis test: two-proportion Z-test (directional, pre-registered)
I pre-register a directional hypothesis (the redesigned flow is expected to improve conversion), which licenses a one-sided test. That licensing only works if the direction was chosen before looking at the results; choosing "one-sided" after seeing which way the data leans is p-hacking with extra steps. So I report the two-sided p-value alongside the one-sided one. With an effect this modest, the gap between the two is the whole story.

- Null $H_0$: $p_{variant} = p_{control}$ (the new onboarding changes nothing).
- Alternative $H_1$: $p_{variant} > p_{control}$ (the new onboarding improves conversion).
- Significance level $\alpha = 0.05$.

In [4]:
control_results = df_ab[df_ab['OnboardingVersion'] == 'Control']['Has_Subscribed']
variant_results = df_ab[df_ab['OnboardingVersion'] == 'Variant']['Has_Subscribed']

# Order variant FIRST so a positive z and the 'larger' alternative both read in
# the direction of the pre-registered hypothesis (variant improves conversion).
successes = [variant_results.sum(), control_results.sum()]
nobs = [variant_results.count(), control_results.count()]

z_stat, p_one = proportions_ztest(successes, nobs=nobs, alternative='larger')
_, p_two = proportions_ztest(successes, nobs=nobs)  # two-sided, reported for honesty
(lo_var, lo_con), (hi_var, hi_con) = proportion_confint(successes, nobs=nobs, alpha=0.05)

lift = variant_results.mean() - control_results.mean()
print(f"[*] Observed lift: {lift*100:+.2f} pp  ({lift/control_results.mean()*100:+.1f}% relative)")
print(f"[*] Z-statistic: {z_stat:.3f}")
print(f"[*] P-value  one-sided (H1: variant > control): {p_one:.4g}")
print(f"[*] P-value  two-sided (reported for honesty):  {p_two:.4g}")
print("-------------------------------------------")
print(f"[+] Variant 95% CI: [{lo_var:.4f}, {hi_var:.4f}]")
print(f"[-] Control 95% CI: [{lo_con:.4f}, {hi_con:.4f}]")

[*] Observed lift: +2.14 pp  (+7.9% relative)
[*] Z-statistic: 1.681
[*] P-value  one-sided (H1: variant > control): 0.04634
[*] P-value  two-sided (reported for honesty):  0.09268
-------------------------------------------
[+] Variant 95% CI: [0.2735, 0.3092]
[-] Control 95% CI: [0.2526, 0.2873]


### 4. Interpretation and Uncertainty Range Visualization
The frequentist decision rule is blunt: if $p < \alpha$, reject $H_0$. Blunt rules still need a picture next to them, because a p-value alone won't tell you how close the two confidence intervals actually sit.

In [5]:
fig, ax = plt.subplots(figsize=(8, 5))

rates = [control_results.mean(), variant_results.mean()]
errors = [(hi_con - lo_con) / 2, (hi_var - lo_var) / 2]

bars = ax.bar(['Control Group', 'Variant Group'], rates, yerr=errors, capsize=10,
              color=['#78909c', '#29b6f6'], alpha=0.85)
ax.set_title("A/B Conversion Effectiveness (95% Confidence Intervals)", fontsize=14, pad=15)
ax.set_ylabel("Average Conversion Rate")

for index, bar in enumerate(bars):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() / 2,
            f"{rates[index]*100:.2f}%", ha='center', color='white', fontweight='bold', fontsize=14)

# Three-way verdict: the honest case is the BORDERLINE one between the two tests.
if p_one < 0.05 and p_two < 0.05:
    verdict, col = "Reject H0 under both one- and two-sided tests", 'lime'
elif p_one < 0.05:
    verdict = f"Borderline — significant one-sided (p={p_one:.3f}) but NOT two-sided (p={p_two:.3f})"
    col = '#FFD54F'
else:
    verdict, col = "Fail to reject H0 (inconclusive)", 'yellow'
plt.suptitle("\nResult: " + verdict, color=col)

plt.show()

### Synthesis — a marginal result, read honestly
The variant lifts conversion by about 2 percentage points (roughly 8% relative), and the verdict sits on the knife edge: one-sided p ≈ 0.046 clears the α = 0.05 bar, two-sided p ≈ 0.093 does not. Three things are worth saying plainly.

This is a synthetic experiment, and I planted the lift myself, so the value here is the method, not the finding. More consequentially, the test choice decides the call: a pre-registered directional hypothesis licenses the one-sided test, under which this result clears 0.05, but switching to one-sided *after* seeing the data, purely to scrape under the line, is a classic p-hacking move dressed up as rigor. Reporting the two-sided p alongside is what keeps the analysis honest, and that number says the evidence is not conclusive. Which means a marginal result like this one is a "maybe," not a "ship it" — the responsible read of p ≈ 0.05 on a ~2 pp effect is promising and under-powered, worth confirming with more data, not a green light. A power calculation would say exactly how many more users that confirmation needs.

The causal reading (that the new flow *caused* the lift) holds only because assignment is randomized by construction. On observational data, the same 2 pp gap could just as easily be confounding: say, the variant happening to reach a stickier mix of devices. That distinction is the fine print separating "I ran a Z-test" from understanding what the test does and does not tell you.